# Project 2 Part I (Core)

*Project 2: Initial Analysis and Problem Selection*

---

## Objective

Perform an initial Exploratory Data Analysis (EDA) on at least four datasets, identify and diagnose potential issues, and select a specific problem to address (regression, classification, clustering, or forecasting). Submit a repository containing the selected dataset, the initial EDA, and a description of the chosen problem.

---

# Part I: Dataset Search and Analysis

## Problem Statement

Develop a classification model capable of predicting whether an aircraft registration corresponds to commercial or private use based on aircraft characteristics such as manufacturer, model, and aircraft type.

# Dataset Description

The dataset contains information about aircraft registrations in Chile, including manufacturer, model, aircraft type, registration details, and operational characteristics.

## Dataset Source

In this project, we analyze the **Aircraft registrations in the National Aircraft Registry (R.N.A.)** dataset, obtained from the Chilean Government Open Data Portal. The dataset contains information about aircrafts registrations until 2026, April 30th, including features such as use, brand, model, kind of aircraft, etc.

The dataset can be accessed and downloaded from the following link:

https://datos.gob.cl/dataset/aeronaves-inscritas-en-el-r-n-a-al-30-abr-2026-xlsx

## Dataset Information

- Number of observations: **2010**
- Number of features: **7**
- Target variable: `USO AERONAVE` 

## Data Dictionary

| Variable              | Data Type   | Description                                                                                                                               | Example                                                |
| --------------------- | ----------- | ----------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------ |
| `ID`                  | Integer     | Sequential record identifier assigned to each aircraft registration record. This column serves as an index and has no predictive meaning. | `1`                                                    |
| `MATRICULA`           | String      | Unique aircraft registration code assigned in Chile. Similar to a vehicle license plate, it identifies a specific aircraft.               | `CCAAA`                                                |
| `USO AERONAVE`        | Categorical | Operational purpose of the aircraft. Indicates whether the aircraft is used for commercial or private activities.                         | `COMERCIAL`, `PRIVADO`                                 |
| `MARCA`               | Categorical | Aircraft manufacturer or brand.                                                                                                           | `BELL`, `CESSNA`, `PIPER`, `AIR TRACTOR`               |
| `MODELO`              | Categorical | Specific model produced by the manufacturer.                                                                                              | `505`, `407`, `AT-802`, `PA-28-161`                    |
| `TIPO DE AERONAVE`    | Categorical | Aircraft classification based on design and operation.                                                                                    | `AVION`, `HELICOPTERO`, `PLANEADOR`                    |
| `NOMBRE DEL OPERADOR` | Categorical | Name of the individual, company, organization, or institution responsible for operating the aircraft.                                     | `TRANSPORTES COSTEROS SPA`, `PUBLICITARIA PUBLI G SPA` |

---

# Data Loading and Inspection

## Import Libraries

In [67]:
# Standard library imports
import math
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

### Dataset Availability and Path Setup

In [68]:
data_dir = Path("../datasets")

# List available files (for verification)
available_files = list(data_dir.iterdir())
print("Available files:", available_files)
file_name = "1_aircrafts_chile_30042026.csv"
dataset_path = data_dir / file_name

Available files: [WindowsPath('../datasets/1_aircrafts_chile_30042026.csv'), WindowsPath('../datasets/2_earthquakes_chile.csv'), WindowsPath('../datasets/3_healthy_diet.csv'), WindowsPath('../datasets/4_temp_chile_2014.csv'), WindowsPath('../datasets/README.md')]


In [69]:
# Validate dataset existence
if not dataset_path.exists():
    raise FileNotFoundError(f"{file_name} not found in {data_dir}")

print("Dataset found at:", dataset_path)

Dataset found at: ..\datasets\1_aircrafts_chile_30042026.csv


## Load Dataset

In [70]:
df = pd.read_csv(dataset_path)

## Dataset Overview


### First and Last Records

In [71]:
df.head()

,N�,MATRICULA,USO AERONAVE,MARCA,MODELO,TIPO DE AERONAVE,NOMBRE DEL OPERADOR
0,1,CCAAA,COMERCIAL,BELL,505,HELICOPTERO,PUBLICITARIA PUBLI G SPA
1,2,CCAAB,PRIVADO,BELL,407,HELICOPTERO,TRANSPORTES COSTEROS SPA
2,3,CCAAC,PRIVADO,WERTH-RANS,S-19,AVION,"WERTH STEINERT, RENATO EDUARDO"
3,4,CCAAE,PRIVADO,SCHEMPP-HIRTH,NIMBUS-4DM,PLANEADOR,"KAUFMANN RITSCHKA, ALEXANDER"
4,5,CCAAF,COMERCIAL,AIR TRACTOR,AT-802,AVION,MARTINEZ RIDAO CHILE LIMITADA


In [72]:
df.tail()

,N�,MATRICULA,USO AERONAVE,MARCA,MODELO,TIPO DE AERONAVE,NOMBRE DEL OPERADOR
2005,1950,CCSZH,PRIVADO,CESSNA,150G,AVION,CLUB A�REO UNIVERSIDAD DE CONCEPCI�N
2006,1951,CCTCA,COMERCIAL,CESSNA,152,AVION,MEGAPARTS S.A.
2007,1952,CCTHJ,PRIVADO,CESSNA,U206G,AVION,AEROVIMATH SPA
2008,1953,CCTHL,PRIVADO,CESSNA,172H,AVION,AERO HUAIRAVO LTDA.
2009,1954,CCTHM,PRIVADO,PIPER,PA-28-161,AVION,"DONOSO MARDONES, CAMILA Y TRONCOSO TRONCOSO, C..."


### Verify Structure

In [73]:
print(f"{file_name}")
print(f"Shape: {df.shape}")
print(f"→ {df.shape[0]} rows, {df.shape[1]} columns\n")

1_aircrafts_chile_30042026.csv
Shape: (2010, 7)
→ 2010 rows, 7 columns



---

## Initial Data Inspection

### Data Quality Issues

A character-encoding issue was identified in the CSV export, affecting special characters in column names. The original identifier column was renamed to ID to ensure consistency throughout the analysis.

In [74]:
df.columns = [
    "ID",
    "MATRICULA",
    "USO AERONAVE",
    "MARCA",
    "MODELO",
    "TIPO DE AERONAVE",
    "NOMBRE DEL OPERADOR"
]

### Inspect Data Types

In [75]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2010 entries, 0 to 2009
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   ID                   2010 non-null   int64
 1   MATRICULA            2010 non-null   str  
 2   USO AERONAVE         2010 non-null   str  
 3   MARCA                2010 non-null   str  
 4   MODELO               2010 non-null   str  
 5   TIPO DE AERONAVE     2010 non-null   str  
 6   NOMBRE DEL OPERADOR  2010 non-null   str  
dtypes: int64(1), str(6)
memory usage: 110.1 KB


In [76]:
df.columns.tolist()

['ID',
 'MATRICULA',
 'USO AERONAVE',
 'MARCA',
 'MODELO',
 'TIPO DE AERONAVE',
 'NOMBRE DEL OPERADOR']

In [77]:
df.nunique()

ID                     1954
MATRICULA              1954
USO AERONAVE              2
MARCA                   216
MODELO                  541
TIPO DE AERONAVE          6
NOMBRE DEL OPERADOR     980
dtype: int64

The dataset contains 2,010 aircraft registration records and 7 variables. No missing values were detected. Most variables are categorical, with aircraft usage (`USO AERONAVE`) presenting only two categories, making it a potential target variable for a binary classification problem. The dataset also contains several high-cardinality variables, particularly aircraft models (541 unique values) and operator names (980 unique values), which may require special preprocessing techniques if used for predictive modeling.

### Descriptive Statistics

In [91]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,2010.0,977.61194,566.730713,1.0,482.25,980.5,1473.75,1954.0


In [92]:
df.describe(include="str").T

,count,unique,top,freq
MATRICULA,2010,1954,CCADB,6
USO AERONAVE,2010,2,PRIVADO,1284
MARCA,2010,216,CESSNA,573
MODELO,2010,541,R44 II,54
TIPO DE AERONAVE,2010,6,AVION,1569
NOMBRE DEL OPERADOR,2010,980,LATAM AIRLINES GROUP S.A.,139


**Conclusion:**
The dataset is almost entirely composed of categorical variables, which suggests that classification tasks may be more appropriate than regression problems.

### Key Findings

1. The dataset contains **2,010 aircraft registration records** and **7 variables**.

2. No missing values have been identified.

3. Most variables are categorical, making the dataset more suitable for classification tasks than regression problems.

4. `USO AERONAVE` contains only two categories and is a strong candidate target variable for binary classification.

5. `MARCA` and `TIPO DE AERONAVE` appear to be potentially informative predictors.

6. `MODELO` and `NOMBRE DEL OPERADOR` exhibit high cardinality and may require special preprocessing techniques.

7. Aircraft registrations are heavily concentrated among certain manufacturers, particularly CESSNA.

8. Airplanes (`AVION`) represent the majority of registered aircraft in the dataset.

---

### Missing Values

In [80]:
df.isnull().sum()

ID                     0
MATRICULA              0
USO AERONAVE           0
MARCA                  0
MODELO                 0
TIPO DE AERONAVE       0
NOMBRE DEL OPERADOR    0
dtype: int64

### Duplicate Records Check

In [82]:
n_duplicated = df.duplicated().sum()
print(f"Exact duplicated rows before cleaning: {n_duplicated}")

Exact duplicated rows before cleaning: 4


In [94]:
df["MATRICULA"].value_counts().loc[
    lambda x: x > 1
]

MATRICULA
CCADB    6
CCASX    4
CCPHC    4
CCPHX    4
CCALW    3
CCAYT    3
CCCWF    3
CCDHC    3
CCPDV    3
CCAFN    2
CCAGV    2
CCAKP    2
CCALN    2
CCALR    2
CCAMG    2
CCAOK    2
CCASA    2
CCATQ    2
CCDCZ    2
CCDDX    2
CCDFN    2
CCLHI    2
CCPBI    2
CCPCP    2
CCPEE    2
CCPFX    2
CCPFY    2
CCPGC    2
CCPIA    2
CCPID    2
CCPIQ    2
CCPLV    2
CCPMU    2
CCPPA    2
CCPQC    2
CCPTH    2
CCPVC    2
CCPVP    2
CCPXX    2
CCPYI    2
CCSFC    2
Name: count, dtype: int64

Although only four exact duplicate records were identified, several aircraft registration numbers appear multiple times. The most frequent registration (`CCADB`) appears six times. This suggests that repeated registrations are not solely explained by duplicated rows and may reflect administrative updates, ownership changes, or other operational factors. Further investigation would be required before using registration-related information in a predictive model.

In [95]:
df[df["MATRICULA"] == "CCADB"]

,ID,MATRICULA,USO AERONAVE,MARCA,MODELO,TIPO DE AERONAVE,NOMBRE DEL OPERADOR
50,51,CCADB,PRIVADO,CESSNA,525,AVION,ALEJANDRO ESTEBAN CANDIA SILVA INVERSIONES E.I...
51,51,CCADB,PRIVADO,CESSNA,525,AVION,ANDES RENTAL SPA
52,51,CCADB,PRIVADO,CESSNA,525,AVION,INVERSIONES E INMOBILIARIA GHC LIMITADA
53,51,CCADB,PRIVADO,CESSNA,525,AVION,"B�CHI BASTIDAS, MAR�A ANA"
54,51,CCADB,PRIVADO,CESSNA,525,AVION,"PHILLIPS GARRET�N, MANUEL JOS�"
55,51,CCADB,PRIVADO,CESSNA,525,AVION,"URRUTIA, MARIA ELEONORA"


Several aircraft registrations appear multiple times in the dataset. An inspection of the most frequent registration (CCADB) revealed that all aircraft characteristics remain identical while only the operator name changes. This suggests that repeated registrations may represent different owners or operators associated with the same aircraft rather than data entry errors. Consequently, repeated registration numbers should not automatically be treated as duplicate records.

---

## Distribution Analysis

### Numeric

#### Histograms

In [83]:
# Select numeric columns
numeric_columns = df.select_dtypes(
    include=np.number
).columns

# Dashboard layout
n_cols = 4
n_rows = math.ceil(len(numeric_columns) / n_cols)

# Create subplot figure
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=numeric_columns
)

# Add histograms
for i, col in enumerate(numeric_columns):
    row = (i // n_cols) + 1
    col_pos = (i % n_cols) + 1

    fig.add_trace(
        go.Histogram(
            x=df[col],
            name=col,
            nbinsx=30
        ),
        row=row,
        col=col_pos
    )

# Update layout
fig.update_layout(
    title="Interactive Histogram Dashboard — Numeric Features",
    height=300 * n_rows,
    width=1400,
    showlegend=False,
    template="plotly_white",
    bargap=0.05
)

# Descriptive axis labels
fig.update_xaxes(title_text="Feature Values")
fig.update_yaxes(title_text="Number of Observations")

# Show dashboard
fig.show()

**Analysis**

The only numerical variable in the dataset is `ID`, which behaves as a sequential identifier. Its histogram shows a nearly uniform distribution across the full range of values, indicating that records were assigned consecutive identifiers rather than representing a measured characteristic of the aircraft. Since `ID` does not contain meaningful business or operational information, it is unlikely to contribute predictive value and will be treated as an identifier rather than a feature for modeling.

### Categorical

#### Categorical Value Consistency

In [96]:
categorical_cols = df.select_dtypes(
    include=["str", "category"]
).columns

for col in categorical_cols:
    print(f"\n{'='*40}")
    print(f"Column: {col}")
    print(f"{'='*40}")

    summary = (
        df[col]
        .value_counts(dropna=False)
        .reset_index()
    )

    summary.columns = [col, 'Count']

    summary['Percentage'] = (
        summary['Count'] / len(df) * 100
    ).round(2)

    print(summary)


Column: MATRICULA
     MATRICULA  Count  Percentage
0        CCADB      6        0.30
1        CCASX      4        0.20
2        CCPHC      4        0.20
3        CCPHX      4        0.20
4        CCALW      3        0.15
...        ...    ...         ...
1949     CCSZH      1        0.05
1950     CCTCA      1        0.05
1951     CCTHJ      1        0.05
1952     CCTHL      1        0.05
1953     CCTHM      1        0.05

[1954 rows x 3 columns]

Column: USO AERONAVE
  USO AERONAVE  Count  Percentage
0      PRIVADO   1284       63.88
1    COMERCIAL    726       36.12

Column: MARCA
             MARCA  Count  Percentage
0           CESSNA    573       28.51
1            PIPER    246       12.24
2           AIRBUS    216       10.75
3         ROBINSON    117        5.82
4             BELL     72        3.58
..             ...    ...         ...
211        SEQUOIA      1        0.05
212      PL - CZAW      1        0.05
213   CAO - JABIRU      1        0.05
214   AEROTEK INC.      1    

In [107]:
categorical_cols = [
    col for col in categorical_cols
    if col not in ["MATRICULA"]
]

MAX_CATEGORIES = 10

n_cols = 2
n_rows = math.ceil(len(categorical_cols) / n_cols)

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=categorical_cols,
    vertical_spacing=0.22
)

for i, col in enumerate(categorical_cols):

    summary = (
        df[col]
        .value_counts(dropna=False)
        .head(MAX_CATEGORIES)
        .reset_index()
    )

    summary.columns = [col, "Count"]

    row = (i // n_cols) + 1
    col_pos = (i % n_cols) + 1

    fig.add_trace(
        go.Bar(
            x=summary[col].astype(str),
            y=summary["Count"],
            text=summary["Count"],
            textposition="outside",
            name=col
        ),
        row=row,
        col=col_pos
    )

    max_count = summary["Count"].max()

    fig.update_yaxes(
        range=[0, max_count * 1.15],
        row=row,
        col=col_pos,
        title_text="Frequency",
        title_standoff=15
    )

fig.update_xaxes(
    title_text="Categories",
    title_standoff=25
    )

fig.update_layout(
    title="Categorical Variables Distribution Dashboard",
    height=500 * n_rows,
    width=1400,
    showlegend=False,
    template="plotly_white"
)

fig.show()

**Overall Finding for the Section**

The dataset is composed almost entirely of categorical variables. Aircraft usage (`USO AERONAVE`) presents a balanced enough distribution to support a binary classification task, while aircraft type, manufacturer, and model provide potentially informative predictors. Several variables exhibit high cardinality, particularly aircraft models and operator names, which should be considered during future preprocessing and feature engineering stages.

### Outlier Detection

In [108]:
if len(numeric_columns) > 0:
    # create boxplots

    # Select numeric columns
    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    # Dashboard layout
    n_cols = 4
    n_rows = math.ceil(len(numeric_columns) / n_cols)

    # Create subplot figure
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=numeric_columns
    )

    # Add boxplots
    for i, col in enumerate(numeric_columns):
        row = (i // n_cols) + 1
        col_pos = (i % n_cols) + 1

        fig.add_trace(
            go.Box(
                y=df[col],
                name=col,
                boxmean=True
            ),
            row=row,
            col=col_pos
        )

    # Update layout
    fig.update_layout(
        title="Interactive Boxplot Dashboard — Numeric Features",
        height=300 * n_rows,
        width=1400,
        showlegend=False,
        template="plotly_white"
    )

    # Add axis labels
    fig.update_yaxes(title_text="Observed Values")

    # Show dashboard
    fig.show()

else:
    print("No numeric variables available for outlier analysis.")

**Analysis**

The `ID` variable does not exhibit unusual observations outside the expected range. Since it represents a sequential record identifier rather than a measured aircraft characteristic, the concept of outliers is not particularly meaningful for this variable. Therefore, no outlier treatment is required, and the variable is expected to be excluded from future predictive modeling.

#### IQR calculation

In [87]:
feature_columns = df.select_dtypes(
    include=np.number
).columns

outlier_summary = []

for col in feature_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    n_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

    outlier_summary.append({
        "Feature": col,
        "Outliers": n_outliers,
        "Percentage": round(n_outliers / len(df) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df

,Feature,Outliers,Percentage
0,ID,0,0.0


The Interquartile Range (IQR) method did not identify any outliers in the `ID` variable. This result is consistent with the histogram and boxplot analyses, which showed a uniformly distributed sequence of identifier values.

Since `ID` is an administrative identifier rather than a business or operational feature, the absence of outliers has limited analytical significance. No outlier treatment is required, and the variable will likely be excluded from future predictive modeling tasks.

No meaningful numerical features are present in this dataset. Therefore, traditional numerical outlier analysis provides limited insight, and the exploratory analysis is primarily focused on categorical variables such as aircraft usage, manufacturer, model, and aircraft type.

---

# Exploratory Data Analysis (EDA)

## Top aircraft manufacturers (`MARCA`)

In [111]:
top_brands = (
    df["MARCA"]
    .value_counts()
    .head(15)
    .reset_index()
)

top_brands.columns = ["Manufacturer", "Count"]

top_brands["Percentage"] = (
    top_brands["Count"] / len(df) * 100
).round(2)

top_brands

,Manufacturer,Count,Percentage
0,CESSNA,573,28.51
1,PIPER,246,12.24
2,AIRBUS,216,10.75
3,ROBINSON,117,5.82
4,BELL,72,3.58
5,BOEING,61,3.03
6,BEECH AIRCRAFT CORPORATION,45,2.24
7,EUROCOPTER,43,2.14
8,BEECHCRAFT CORP.,40,1.99
9,AEROPRAKT LTD.,39,1.94


In [112]:
fig = px.bar(
    top_brands,
    x="Manufacturer",
    y="Count",
    text="Percentage",
    title="Top 15 Aircraft Manufacturers",
)

fig.update_traces(
    texttemplate="%{text}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Manufacturer",
    yaxis_title="Number of Aircraft Registrations",
    height=600
)

fig.show()

The dataset contains aircraft registrations from 216 different manufacturers, indicating substantial diversity in the Chilean aviation sector.

However, registrations are concentrated among a relatively small number of manufacturers. The leading manufacturers account for a significant proportion of all registered aircraft, suggesting that the market is dominated by a few well-established brands.

This concentration may reflect differences in aircraft availability, operational costs, maintenance support, and market preferences.

Because manufacturer information captures important characteristics of the aircraft, `MARCA` may serve as a valuable predictor in future machine learning models aimed at classifying aircraft usage or identifying operational patterns.

In [113]:
brand_usage = pd.crosstab(
    df["MARCA"],
    df["USO AERONAVE"]
)

brand_usage.sort_values(
    by="COMERCIAL",
    ascending=False
).head(15)

USO AERONAVE,COMERCIAL,PRIVADO
MARCA,,
AIRBUS,216,0
CESSNA,74,499
BOEING,60,1
PIPER,56,190
BELL,45,27
EUROCOPTER,37,6
AIR TRACTOR,18,5
AIRBUS HELICOPTERS,14,0
ROBINSON,11,106


Cross-tabulation analysis reveals a strong relationship between aircraft manufacturer and aircraft usage. Major commercial manufacturers such as Airbus and Boeing are overwhelmingly associated with commercial operations, while manufacturers such as Cessna, Piper, and Robinson are primarily linked to private aviation. These results suggest that manufacturer information may provide substantial predictive power for a future classification model aimed at predicting aircraft usage.

## Aircraft Type vs Aircraft Usage

In [114]:
pd.crosstab(
    df["TIPO DE AERONAVE"],
    df["USO AERONAVE"],
    normalize="index"
).round(3)

USO AERONAVE,COMERCIAL,PRIVADO
TIPO DE AERONAVE,,
AVION,0.349,0.651
DIRIGIBLE,0.500,0.500
GIROPL,0.000,1.000
GLOBO,0.059,0.941
HELICOPTERO,0.530,0.470
PLANEADOR,0.000,1.000


Aircraft type exhibits a meaningful relationship with aircraft usage. While some aircraft categories such as gliders and gyrocopters are exclusively associated with private operations, airplanes and helicopters are used in both commercial and private contexts. Helicopters show the most balanced distribution between usage categories, suggesting that aircraft type alone is insufficient to fully explain operational use. Consequently, aircraft type may provide valuable predictive information when combined with other features such as manufacturer and model.

## Aircraft usage distribution (`USO AERONAVE`)

In [115]:
usage_summary = (
    df["USO AERONAVE"]
    .value_counts()
    .reset_index()
)

usage_summary.columns = ["Aircraft Usage", "Count"]

usage_summary["Percentage"] = (
    usage_summary["Count"] / len(df) * 100
).round(2)

usage_summary

,Aircraft Usage,Count,Percentage
0,PRIVADO,1284,63.88
1,COMERCIAL,726,36.12


In [116]:
fig = px.bar(
    usage_summary,
    x="Aircraft Usage",
    y="Count",
    text="Percentage",
    title="Aircraft Usage Distribution"
)

fig.update_traces(
    texttemplate="%{text}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Usage Category",
    yaxis_title="Number of Registrations",
    height=500
)

fig.show()

**Analysis**

The dataset contains two aircraft usage categories: **Private** and **Commercial**.

Private aircraft registrations represent approximately **63.9%** of all observations, while commercial registrations account for **36.1%**. This indicates that private aviation constitutes the majority of registered aircraft in the dataset.

Although the classes are not perfectly balanced, the distribution is not severely skewed. Both categories contain a sufficient number of observations to support supervised machine learning techniques.

The moderate class imbalance should be considered during model evaluation, but it is unlikely to require aggressive resampling strategies.

In [117]:
class_ratio = (
    usage_summary["Count"].max()
    / usage_summary["Count"].min()
)

print(f"Majority/Minority ratio: {class_ratio:.2f}")

Majority/Minority ratio: 1.77


The majority class is only 1.77 times larger than the minority class, indicating a relatively balanced classification problem.

## Cross-tabulations

In [109]:
pd.crosstab(
    df["TIPO DE AERONAVE"],
    df["USO AERONAVE"],
    normalize="index"
).round(3)

USO AERONAVE,COMERCIAL,PRIVADO
TIPO DE AERONAVE,,
AVION,0.349,0.651
DIRIGIBLE,0.500,0.500
GIROPL,0.000,1.000
GLOBO,0.059,0.941
HELICOPTERO,0.530,0.470
PLANEADOR,0.000,1.000


In [118]:
usage_by_type = pd.crosstab(
    df["TIPO DE AERONAVE"],
    df["USO AERONAVE"],
    normalize="index"
).mul(100)

fig = px.bar(
    usage_by_type,
    barmode="stack",
    title="Aircraft Usage Distribution by Aircraft Type",
    labels={
        "value": "Percentage",
        "TIPO DE AERONAVE": "Aircraft Type"
    }
)

fig.update_layout(
    template="plotly_white",
    yaxis_title="Percentage (%)",
    xaxis_title="Aircraft Type"
)

fig.show()

**Analysis**

The distribution of aircraft usage varies considerably across aircraft types.

Airplanes (`AVION`) are predominantly used for private purposes, with approximately 65.1% of registrations classified as private and 34.9% as commercial. In contrast, helicopters (`HELICOPTERO`) show a more balanced distribution, with commercial registrations slightly exceeding private registrations (53.0% versus 47.0%).

Certain aircraft types appear almost exclusively in private operations. Both gliders (`PLANEADOR`) and gyrocopters (`GIROPL`) are registered only for private use in this dataset, while balloons (`GLOBO`) are overwhelmingly private (94.1%).

Airships (`DIRIGIBLE`) display an even distribution between commercial and private use; however, given the very small number of observations, this result should be interpreted with caution.

**Key Findings**
- Aircraft type is associated with aircraft usage.
- Helicopters are the only major category with a near-balanced commercial/private split.
- Gliders and gyrocopters are exclusively private.
- Balloons are overwhelmingly private.
- Airplanes are mostly private but still represent a substantial share of commercial operations.

**Implications for Modeling**

The observed differences suggest that `TIPO DE AERONAVE` contains predictive information regarding aircraft usage. However, the relationship is not deterministic for the most common categories, particularly airplanes and helicopters. Therefore, aircraft type alone is unlikely to perfectly predict the target variable and should be combined with other features such as manufacturer (`MARCA`) and model (`MODELO`) in a future classification model.

---

## Potential Target Variable

Candidate target:
- `USO AERONAVE`

Potential ML task:
- Classification

Rationale:

`USO AERONAVE` was selected as the candidate target variable because it represents a binary classification problem (`COMERCIAL` vs. `PRIVADO`). Exploratory analysis identified meaningful relationships between aircraft usage, manufacturer, and aircraft type, indicating potential predictive power. Additionally, both classes are well represented in the dataset, making it suitable for supervised machine learning classification models.

---

# EDA Findings Summary

## Findings

- The dataset contains **2,010 aircraft registration records** and **7 variables**, primarily describing aircraft characteristics and operators.
- No missing values were detected in any column, indicating a high level of data completeness.
- Four exact duplicate records were identified. Further inspection suggests that repeated registrations may be associated with aircraft ownership changes rather than data quality issues.
- The dataset is predominantly categorical, with only one numerical variable (`ID`), which functions as an identifier and does not provide analytical value for predictive modeling.
- Aircraft registrations are primarily classified as **Private (63.9%)**, while **Commercial (36.1%)** registrations represent a substantial minority.
- The aviation market appears concentrated among a relatively small number of manufacturers. **Cessna**, **Piper**, and **Airbus** account for a significant share of registered aircraft.
- Manufacturer (`MARCA`) shows a strong relationship with aircraft usage. For example, Airbus and Boeing aircraft are overwhelmingly associated with commercial operations, whereas Cessna and Piper are predominantly associated with private use.
- Aircraft type (`TIPO DE AERONAVE`) also exhibits meaningful differences across usage categories. Gliders and gyrocopters are exclusively private, while helicopters are more evenly distributed between commercial and private operations.
- No numerical outliers were detected; however, numerical outlier analysis provides limited insight because the dataset lacks operational numerical features.
- The dataset presents a viable **classification problem**, with `USO AERONAVE` identified as a strong candidate target variable and aircraft characteristics serving as potential predictors.

### Conclusion

Based on the exploratory analysis, the dataset demonstrates sufficient quality, completeness, and predictive potential to support a supervised classification task. The relationship between aircraft usage and variables such as manufacturer, model, and aircraft type suggests that a machine learning model could be developed to predict whether an aircraft is registered for commercial or private use. This dataset will be considered a strong candidate for further analysis in the next phase of the project.

---

# Appendix

### References

- Aircraft registrations in the National Aircraft Registry (R.N.A.) Dataset, Chilean Government Open Data Portal: [Aircrafts Dataset](https://datos.gob.cl/dataset/aeronaves-inscritas-en-el-r-n-a-al-30-abr-2026-xlsx)
- Machine Learning notebooks from the bootcamp repository (`sonda2026` branch)
- Machine Learning notebooks from the shared Assistanship Google Drive folder
- [Pandas Documentation](https://pandas.pydata.org/docs/user_guide/index.html)
- [Plotly Documentation](https://docs.plotly.com/)

### Acknowledgements

I would like to thank my instructors for their guidance, continuous support, and encouragement throughout this project.

I also acknowledge the use of AI-assisted tools for support with debugging, code review, documentation, and learning concepts during the development of the analysis. All final decisions, interpretations, and conclusions presented in this work remain my own.